# Análise e Teste de LSTM - Treinamento Local

Este notebook implementa localmente todo o workflow de treinamento de uma LSTM para predição de movimentos de mercado, seguindo a mesma arquitetura e parâmetros utilizados no projeto WTNPS Trade.

**Objetivo:** Treinar uma LSTM do zero e analisar sua performance através de métricas de classificação.

## 1. Importação de Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import MetaTrader5 as mt5
from datetime import datetime, timedelta
import pytz

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
# Métricas para regressão
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# Correlações (relevantes para regressão, como no notebook de referência)
from scipy.stats import pearsonr, spearmanr

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Bibliotecas importadas com sucesso!")

# Inicializar MT5
if not mt5.initialize():
    print(f"Erro ao inicializar MT5: {mt5.last_error()}")
    raise Exception("Falha na inicialização do MetaTrader 5")

print(f"MT5 inicializado com sucesso!")
print(f"Versão: {mt5.version()}")

Bibliotecas importadas com sucesso!
MT5 inicializado com sucesso!
Versão: (500, 5370, '17 Oct 2025')


## 2. Parâmetros de Configuração

Definição local de todos os parâmetros utilizados no treinamento.

In [ ]:
# Parâmetros do ativo e dados
TICKER = "WDO$"  # Ou "WDO$" conforme preferência
TIMEFRAME = mt5.TIMEFRAME_H1  # Timeframe H1
START_DATE = datetime(2023, 1, 1)  # Data de início
END_DATE = datetime(2025, 9, 30)  # Data de fim

# Parâmetros da LSTM
LOOKBACK = 60  # Janela de lookback (60 candles anteriores)
LSTM_UNITS = 50  # Unidades LSTM
EPOCHS = 50  # Número de épocas
BATCH_SIZE = 32  # Tamanho do batch
TARGET_PERIOD = 30  # Período de previsão (30 candles)

# Parâmetros de divisão de dados
TEST_SIZE = 0.2  # 20% para teste
VALIDATION_SPLIT = 0.1  # 10% do treino para validação

# Timezone
TIMEZONE_SP = pytz.timezone('Etc/UTC')

print(f"Configuração:")
print(f"  Ticker: {TICKER}")
print(f"  Período: {START_DATE.date()} a {END_DATE.date()}")
print(f"  Lookback: {LOOKBACK}")
print(f"  LSTM Units: {LSTM_UNITS}")
print(f"  Epochs: {EPOCHS}")

## 3. Coleta de Dados

In [ ]:
# Converter datas para UTC (MT5 trabalha em UTC)
start_date_utc = TIMEZONE_SP.localize(START_DATE).astimezone(pytz.UTC)
end_date_utc = TIMEZONE_SP.localize(END_DATE).astimezone(pytz.UTC)

print(f"\nBuscando dados de {TICKER}...")
print(f"  Período UTC: {start_date_utc} a {end_date_utc}")

# Buscar dados históricos
rates = mt5.copy_rates_range(TICKER, TIMEFRAME, start_date_utc, end_date_utc)

if rates is None or len(rates) == 0:
    print(f"Erro ao buscar dados: {mt5.last_error()}")
    mt5.shutdown()
    raise Exception("Nenhum dado retornado do MT5")

print(f"\n✓ {len(rates)} candles coletados com sucesso!")

# Converter para DataFrame
df_raw = pd.DataFrame(rates)
df_raw['time'] = pd.to_datetime(df_raw['time'], unit='s', utc=True)
df_raw['time'] = df_raw['time'].dt.tz_convert(TIMEZONE_SP)
df_raw.set_index('time', inplace=True)

# Renomear colunas para padrão do projeto
df_raw.columns = ['open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume']
df_raw.rename(columns={'tick_volume': 'volume'}, inplace=True)

print(f"\nPrimeiros registros:")
print(df_raw.head())
print(f"\nÚltimos registros:")
print(df_raw.tail())

# Desligar MT5
mt5.shutdown()
print("\nMT5 desconectado.")

## 4. Criação de Features (Indicadores Técnicos)

Implementação local das mesmas features utilizadas na estratégia LSTM do projeto.

In [ ]:
def create_features(df):
    """
    Cria indicadores técnicos como features.
    Mesma lógica de LSTMStrategy.define_features()
    """
    df = df.copy()
    
    # Médias Móveis
    df['ema_9'] = df['close'].ewm(span=9, adjust=False).mean()
    df['sma_20'] = df['close'].rolling(window=20).mean()
    # df['sma_50'] = df['close'].rolling(window=50).mean()
    # df['sma_200'] = df['close'].rolling(window=200).mean()

    # Distância do preço à média móvel
    # df['dist_ema_9'] = df['close'] - df['ema_9']
    
    df['dist_sma_20'] = df['close'] - df['sma_20']
    
    # df['dist_sma_50'] = df['close'] - df['sma_50']
    # df['dist_sma_200'] = df['close'] - df['sma_200']

    # Volatilidade (desvio padrão dos retornos)
    df['returns'] = df['close'].pct_change()
    volatility_window = min(21, len(df) - 1) if len(df) > 1 else 1
    if volatility_window > 0:
        df['volatility'] = df['returns'].rolling(window=volatility_window).std()
    else:
        df['volatility'] = 0.0
    
    # RSI (Índice de Força Relativa)
    """
    rsi_period = 14
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=rsi_period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=rsi_period).mean()
    rs = gain / loss.replace(0, 1e-6)
    df['rsi'] = 100 - (100 / (1 + rs))
    df['rsi'] = df['rsi'].fillna(50)
    """

    # Remove colunas auxiliares
    df = df.drop(columns=['returns'], errors='ignore')
    
    # Preenche NaNs
    df.ffill(inplace=True)
    df.bfill(inplace=True)
    
    return df

print("Criando features...")
df_features = create_features(df_raw)

# Features que serão usadas no modelo
FEATURE_NAMES = ['ema_9', 'sma_20', 'dist_sma_20', 'volume', 'volatility']

print(f"\n✓ Features criadas: {FEATURE_NAMES}")
print(f"\nEstatísticas das features:")
print(df_features[FEATURE_NAMES].describe())

## 5. Criação do Target (Rótulo)

Target binário: 1 se o preço sobe no próximo período, 0 caso contrário.

In [ ]:
def create_target(df, target_period=TARGET_PERIOD):
    """
    Cria target contínuo: retorno futuro percentual em relação ao close atual.
    target = (close_{t+periodo} / close_t) - 1.0
    """
    df = df.copy()
    df['future_close'] = df['close'].shift(-target_period)
    df['target'] = (df['future_close'] / df['close']) - 1.0
    return df['target']

print("Criando target (retorno futuro)...")
df_features['target'] = create_target(df_features, TARGET_PERIOD)

# Remove linhas com NaN (últimas linhas sem target)
df_clean = df_features[FEATURE_NAMES + ['target']].dropna()

print(f"\n✓ Target criado (regressão: retorno contínuo)")
print(f"\nEstatísticas do target (retorno):")
print(df_clean['target'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print(f"\nTotal de amostras após limpeza: {len(df_clean)}")

## 6. Preparação dos Dados

Divisão em treino/teste, normalização e criação de sequências para LSTM.

In [ ]:
# Separar features e target
X = df_clean[FEATURE_NAMES].values
y = df_clean['target'].values

print(f"Shape original - X: {X.shape}, y: {y.shape}")

# Divisão treino/teste (temporal - não embaralhado)
split_idx = int(len(X) * (1 - TEST_SIZE))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"\nDivisão treino/teste:")
print(f"  Treino: {len(X_train)} amostras ({(1-TEST_SIZE)*100:.0f}%)")
print(f"  Teste: {len(X_test)} amostras ({TEST_SIZE*100:.0f}%)")

# Normalização (fit apenas no treino!)
scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Dados normalizados")

In [ ]:
def create_sequences(X_data, y_data, lookback):
    """
    Cria sequências para LSTM.
    Mesma função de lstm.py
    """
    X, y = [], []
    for i in range(len(X_data) - lookback):
        X.append(X_data[i:(i + lookback), :])
        y.append(y_data[i + lookback])
    return np.array(X), np.array(y)

print(f"Criando sequências com lookback={LOOKBACK}...")

# Criar sequências
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, LOOKBACK)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test, LOOKBACK)

print(f"\n✓ Sequências criadas")
print(f"  X_train_seq: {X_train_seq.shape} - (amostras, lookback, features)")
print(f"  y_train_seq: {y_train_seq.shape}")
print(f"  X_test_seq: {X_test_seq.shape}")
print(f"  y_test_seq: {y_test_seq.shape}")

# Verificar distribuição do target nas sequências
print(f"\nDistribuição do target no treino (sequências):")
unique, counts = np.unique(y_train_seq, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  Classe {int(val)}: {count} ({count/len(y_train_seq)*100:.2f}%)")

print(f"\nDistribuição do target no teste (sequências):")
unique, counts = np.unique(y_test_seq, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  Classe {int(val)}: {count} ({count/len(y_test_seq)*100:.2f}%)")

## 7. Construção do Modelo LSTM

Arquitetura idêntica à utilizada em `LSTMWrapper._build_model()`.

In [ ]:
def build_lstm_model(lookback, n_features, lstm_units=50):
    """
    Constrói modelo LSTM para regressão de retornos.
    Arquitetura baseada em LSTMWrapper._build_model(), ajustada para saída linear.
    """
    model = Sequential()

    # Primeira camada LSTM com return_sequences=True
    model.add(LSTM(units=lstm_units, return_sequences=True,
                   input_shape=(lookback, n_features)))
    model.add(Dropout(0.2))

    # Segunda camada LSTM
    model.add(LSTM(units=lstm_units, return_sequences=False))
    model.add(Dropout(0.2))

    # Camadas Dense
    model.add(Dense(units=25))
    model.add(Dense(units=1, activation='linear'))  # Saída contínua (retorno)

    # Compilação para regressão
    model.compile(optimizer='adam',
                  loss='mse',
                  metrics=['mae'])

    return model

# Construir o modelo
n_features = len(FEATURE_NAMES)
model = build_lstm_model(LOOKBACK, n_features, LSTM_UNITS)

print("✓ Modelo LSTM de regressão construído")
print("\nArquitetura do modelo:")
model.summary()

## 8. Treinamento do Modelo

Treino com Early Stopping para evitar overfitting.

In [ ]:
# Callback Early Stopping
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

print(f"Iniciando treinamento...")
print(f"  Épocas: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Validation split: {VALIDATION_SPLIT}")
print(f"  Early stopping: patience=10\n")

# Treinar o modelo
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=[early_stopping],
    verbose=1
)

print("\n✓ Treinamento concluído!")

## 9. Visualização do Treinamento

Gráficos de loss e accuracy durante o treinamento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss (MSE)
axes[0].plot(history.history['loss'], label='Treino', linewidth=2)
if 'val_loss' in history.history:
    axes[0].plot(history.history['val_loss'], label='Validação', linewidth=2)
axes[0].set_xlabel('Época', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Evolução do Loss (MSE)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# MAE
if 'mae' in history.history:
    axes[1].plot(history.history['mae'], label='Treino', linewidth=2)
if 'val_mae' in history.history:
    axes[1].plot(history.history['val_mae'], label='Validação', linewidth=2)
axes[1].set_xlabel('Época', fontsize=12)
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].set_title('Evolução do MAE', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Informações sobre early stopping
stopped_epoch = len(history.history['loss'])
print(f"\nTreinamento parou na época: {stopped_epoch}/{EPOCHS}")
print(f"Melhor val_loss (MSE): {min(history.history.get('val_loss', [np.nan])):.6f}")
if 'val_mae' in history.history:
    print(f"Melhor val_mae: {min(history.history['val_mae']):.6f}")

## 10. Predições no Conjunto de Teste

In [ ]:
print("Gerando predições no conjunto de teste...")

# Predições contínuas (retornos)
y_pred = model.predict(X_test_seq).flatten()

print(f"\n✓ Predições geradas")
print(f"  Shape y_test_seq: {y_test_seq.shape}")
print(f"  Shape y_pred: {y_pred.shape}")
print(f"\nEstatísticas das predições (retorno):")
print(pd.Series(y_pred).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

## 11. Análise de Performance - Métricas de Regressão

Avaliação detalhada da performance do modelo em termos de erro e correlação.

In [ ]:
# Métricas de regressão
mse = mean_squared_error(y_test_seq, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_seq, y_pred)
try:
    r2 = r2_score(y_test_seq, y_pred)
except Exception:
    r2 = np.nan

# MAPE manual para evitar dependências
mape = np.mean(np.abs((y_test_seq - y_pred) / np.where(y_test_seq != 0, np.abs(y_test_seq), 1e-8))) * 100

# Correlações (com p-valor)
try:
    pearson, pearson_p = pearsonr(y_test_seq, y_pred)
except Exception:
    pearson, pearson_p = (np.nan, np.nan)

try:
    spearman, spearman_p = spearmanr(y_test_seq, y_pred)
except Exception:
    spearman, spearman_p = (np.nan, np.nan)

print("="*60)
print(" "*15 + "MÉTRICAS DE PERFORMANCE (REGRESSÃO)")
print("="*60)
print(f"\nMSE:                        {mse:.8f}")
print(f"RMSE:                       {rmse:.8f}")
print(f"MAE:                        {mae:.8f}")
print(f"MAPE:                       {mape:.4f}%")
print(f"R² (Coef. de Determinação): {r2:.6f}")
print(f"Correlação Pearson:         {pearson:.6f} (p={pearson_p:.2e})")
print(f"Correlação Spearman:        {spearman:.6f} (p={spearman_p:.2e})")
print("\n" + "="*60)

print("\n📊 INTERPRETAÇÃO DAS MÉTRICAS:")
print("• RMSE e MAE indicam o erro médio em unidades de retorno (decimais).")
print("• R² próximo de 1 indica boa explicação da variância; próximo de 0 indica fraca.")
print("• Correlações positivas indicam alinhamento entre real e predito; p-valor pequeno indica significância.")

## 12. Dispersão Predito vs Real

In [ ]:
# Dispersão: Predito vs Real
plt.figure(figsize=(8, 7))
plt.scatter(y_test_seq, y_pred, alpha=0.5, edgecolor='k', linewidth=0.3)
min_val = float(min(np.min(y_test_seq), np.min(y_pred)))
max_val = float(max(np.max(y_test_seq), np.max(y_pred)))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Referência y=x')
plt.xlabel('Retorno Real', fontsize=12, fontweight='bold')
plt.ylabel('Retorno Predito', fontsize=12, fontweight='bold')
plt.title('Dispersão: Predito vs Real', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Análise simples dos resíduos
residuals = y_test_seq - y_pred
print("\n📊 RESÍDUOS - Estatísticas básicas:")
print(pd.Series(residuals).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

## 13. Relatório de Resíduos

In [ ]:
# Relatório de Resíduos
residuals = y_test_seq - y_pred

print("\n" + "="*60)
print(" "*10 + "RELATÓRIO DE RESÍDUOS (REGRESSÃO)")
print("="*60 + "\n")

res_series = pd.Series(residuals)
print("Estatísticas dos resíduos:")
print(res_series.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

# Histograma dos resíduos
plt.figure(figsize=(10, 6))
plt.hist(residuals, bins=60, color='steelblue', edgecolor='black', alpha=0.75)
plt.axvline(x=0.0, color='red', linestyle='--', linewidth=2, label='Zero')
plt.xlabel('Resíduo (Real - Predito)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.title('Distribuição dos Resíduos', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📝 Observações:")
print("• Distribuição simétrica e centrada em 0 indica viés baixo.")
print("• Caudas longas indicam eventos extremos (outliers) — considerar robustez.")

## 14. Distribuição dos Retornos - Real vs Predito

In [ ]:
# Distribuição dos retornos: Real vs Predito
plt.figure(figsize=(10, 7))
plt.hist(y_test_seq, bins=60, alpha=0.6, label='Real', color='green', edgecolor='black')
plt.hist(y_pred, bins=60, alpha=0.6, label='Predito', color='orange', edgecolor='black')
plt.xlabel('Retorno', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.title('Distribuição dos Retornos - Real vs Predito', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Distribuições comparadas — procure alinhamento geral; desvios indicam viés ou sub/superdispersão.")

## 15. Análise de Resíduos - Complementar

In [ ]:
# Análise de Resíduos Complementar
residuals = y_test_seq - y_pred

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Resíduos vs Predito
axes[0].scatter(y_pred, residuals, alpha=0.5, edgecolor='k', linewidth=0.3)
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Retorno Predito', fontsize=12)
axes[0].set_ylabel('Resíduo (Real - Predito)', fontsize=12)
axes[0].set_title('Resíduos vs Predito', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Resíduos ao longo das amostras
axes[1].plot(residuals, color='purple', linewidth=1)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Índice da Amostra (teste)', fontsize=12)
axes[1].set_ylabel('Resíduo', fontsize=12)
axes[1].set_title('Resíduos ao Longo do Tempo (Teste)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 ANÁLISE:")
print(f"  Média dos resíduos: {np.mean(residuals):.8f}")
print(f"  Desvio padrão:      {np.std(residuals):.8f}")
print(f"  Mediana:            {np.median(residuals):.8f}")

## 16. Resumo Final e Conclusões

In [ ]:
print("\n" + "="*70)
print(" "*20 + "RESUMO FINAL DA ANÁLISE")
print("="*70)

print(f"\n📋 CONFIGURAÇÃO DO MODELO:")
print(f"  • Ticker: {TICKER}")
print(f"  • Timeframe: M5")
print(f"  • Período: {START_DATE.date()} a {END_DATE.date()}")
print(f"  • Total de candles: {len(df_raw)}")
print(f"  • Lookback: {LOOKBACK}")
print(f"  • LSTM Units: {LSTM_UNITS}")
print(f"  • Features utilizadas: {len(FEATURE_NAMES)}")
print(f"  • Épocas treinadas: {stopped_epoch}/{EPOCHS}")

print(f"\n📊 RESULTADOS DE PERFORMANCE (REGRESSÃO):")
print(f"  • MSE:    {mse:.8f}")
print(f"  • RMSE:   {rmse:.8f}")
print(f"  • MAE:    {mae:.8f}")
print(f"  • MAPE:   {mape:.4f}%")
print(f"  • R²:     {r2:.6f}")
print(f"  • Corr (Pearson):  {pearson:.6f}")
print(f"  • Corr (Spearman): {spearman:.6f}")

print(f"\n✅ ANÁLISE QUALITATIVA:")
print("  • RMSE/MAE baixos indicam bom ajuste em magnitude de retorno.")
print("  • R² e correlações positivas sugerem alinhamento direcional.")
print("  • Se MAPE elevado, modelo erra muito em retornos pequenos — avalie robustez.")

print(f"\n⚠️  CONSIDERAÇÕES:")
print(f"  • Retornos têm ruído alto; avalie estabilidade por janelas temporais.")
print(f"  • Teste regularização, tuning de hiperparâmetros e features adicionais (ex.: RSI).")
print(f"  • Realize backtest com custos e slippage antes de uso real.")

print("\n" + "="*70 + "\n")

## 17. Análise de Predições ao Longo do Tempo (Opcional)

Visualização das predições versus valores reais em uma amostra temporal.

In [ ]:
# Plotar uma amostra de predições vs real (retornos)
sample_size = min(200, len(y_test_seq))  # Últimos 200 pontos
sample_indices = np.arange(len(y_test_seq) - sample_size, len(y_test_seq))

plt.figure(figsize=(15, 6))
plt.plot(sample_indices, y_test_seq[-sample_size:], 'o-', label='Real', 
         alpha=0.7, markersize=4, linewidth=1.5, color='blue')
plt.plot(sample_indices, y_pred[-sample_size:], 's-', label='Predito', 
         alpha=0.7, markersize=4, linewidth=1.5, color='red')
plt.xlabel('Índice Temporal (amostras)', fontsize=12)
plt.ylabel('Retorno (decimal)', fontsize=12)
plt.title(f'Comparação Real vs Predito (Retornos) - Últimas {sample_size} Amostras', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calcular RMSE na amostra
sample_rmse = np.sqrt(mean_squared_error(y_test_seq[-sample_size:], y_pred[-sample_size:]))
print(f"\n✓ RMSE na amostra plotada: {sample_rmse:.8f}")

## 🎯 Conclusão

Este notebook implementou e analisou uma LSTM para predição de movimentos de mercado, seguindo fielmente a arquitetura do projeto WTNPS Trade.

**Próximos passos sugeridos:**
1. Experimentar com diferentes lookbacks (30, 90, 120)
2. Testar diferentes conjuntos de features
3. Ajustar hiperparâmetros (LSTM units, dropout, learning rate)
4. Implementar validação temporal (walk-forward)
5. Integrar com análise de setup (filtros de entrada)
6. Realizar backtest completo com gestão de risco